# Phase 2: FGSM Attack & Principled Defenses on Florence-2

## Overview
This notebook implements a **clean, fair** evaluation of Florence-2-Base's adversarial robustness under FGSM attack, with properly designed defenses.

### What's Different from Phase 1
| Issue in Phase 1 | Fix in Phase 2 |
|---|---|
| JPEG compression applied *before* attack (useless) | Applied *after* attack, before re-inference |
| Gaussian noise injected 3-4 times cumulatively | Single application, fixed sigma |
| Prompt ensemble mixed `<OD>` with free-form text | Single `<OD>` prompt, no mixing |
| Score recomputation `0.6 + 0.2*area + 0.15*center` | Raw model scores used as-is |
| "No defense" baseline used multi-scale + NMS extras | Identical inference pipeline for all conditions |
| Clean mAP with defense never measured | Measured for every defense (defense cost) |

### Defenses Implemented (All Applied After Attack, Before Re-Inference)
1. **JPEG Compression** -- Dziugaite et al., "A Study of the Effect of JPG Compression on Adversarial Images," 2016
2. **Gaussian Blur** -- Xu et al., "Feature Squeezing: Detecting Adversarial Examples," NDSS 2018
3. **Spatial Smoothing (Median Filter)** -- Xu et al., "Feature Squeezing," NDSS 2018
4. **DiffPure (Diffusion Purification)** -- Nie et al., "Diffusion Models for Adversarial Purification," ICML 2022
5. **SVD Spectral Filtering** -- Inspired by Darabi et al., "EigenShield," arXiv:2502.14976, 2025

### Evaluation Protocol
- All conditions use **identical** inference: single `<OD>` prompt, standard NMS, raw model scores.
- The **only variable** is what happens to the image before inference.

## 1. Setup and Imports

In [ ]:
import os
import json
import torch
import numpy as np
from PIL import Image, ImageFilter
from io import BytesIO
from tqdm.auto import tqdm
from transformers import AutoProcessor, AutoModelForCausalLM
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from torchvision import transforms
import GPUtil
import time
import warnings
import gc
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# Optional: DiffPure requires diffusers
# Install with: pip install diffusers
DIFFPURE_AVAILABLE = False
try:
    from diffusers import DDPMScheduler, UNet2DModel
    DIFFPURE_AVAILABLE = True
    print("diffusers available -- DiffPure defense enabled")
except ImportError:
    print("diffusers not installed -- DiffPure defense disabled")
    print("Install with: pip install diffusers")

print("All core imports successful.")

## 2. Configuration

Set all experiment parameters here. Change `NUM_IMAGES` to `None` for full evaluation.

In [ ]:
# ============================================================
# CONFIGURATION -- Edit these parameters
# ============================================================

# Dataset paths
IMAGE_DIR = "./Dataset/coco/images/val2017"
ANN_FILE = "./Dataset/coco/annotations/annotations/instances_val2017.json"

# Number of images (None = all 5000)
NUM_IMAGES = 500  # Use 50 for debugging, 500 for quick validation, None for full eval

# FGSM epsilon values to test
EPSILONS = [0.003, 0.01, 0.03]

# Defenses to run (set False to skip)
RUN_JPEG_DEFENSE = True
RUN_BLUR_DEFENSE = True
RUN_MEDIAN_DEFENSE = True
RUN_DIFFPURE_DEFENSE = DIFFPURE_AVAILABLE
RUN_SVD_DEFENSE = True

# Defense parameters
JPEG_QUALITY = 75          # Dziugaite et al., 2016: quality 75 is effective
BLUR_SIGMA = 1.0           # Xu et al., 2018: sigma 1.0 for Gaussian blur
MEDIAN_KERNEL = 3          # Xu et al., 2018: 3x3 median filter
DIFFPURE_TIMESTEPS = 100   # Nie et al., 2022: ~100 forward steps
SVD_KEEP_RATIO = 0.9       # Keep top 90% singular values (filter adversarial high-freq)

# NMS threshold (same for ALL conditions)
NMS_IOU_THRESHOLD = 0.5

# Output directory
OUTPUT_DIR = "./results_phase2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Configuration loaded.")
print(f"  Images: {NUM_IMAGES or 'all'}")
print(f"  Epsilons: {EPSILONS}")
print(f"  Defenses: JPEG={RUN_JPEG_DEFENSE}, Blur={RUN_BLUR_DEFENSE}, "
      f"Median={RUN_MEDIAN_DEFENSE}, DiffPure={RUN_DIFFPURE_DEFENSE}, SVD={RUN_SVD_DEFENSE}")

## 3. Load Model and Dataset

In [ ]:
# Device setup
def get_free_gpu():
    gpus = GPUtil.getAvailable(order='memory', limit=1)
    return f"cuda:{gpus[0]}" if gpus else "cpu"

device = torch.device(get_free_gpu())
torch_dtype = torch.float16 if device.type == "cuda" else torch.float32
print(f"Device: {device}, Dtype: {torch_dtype}")

# Load Florence-2
model_name = "microsoft/Florence-2-base"
revision = "refs/pr/26"

print("Loading Florence-2-Base model...")
model = AutoModelForCausalLM.from_pretrained(
    model_name, revision=revision,
    torch_dtype=torch_dtype, trust_remote_code=True
).to(device)
processor = AutoProcessor.from_pretrained(
    model_name, revision=revision, trust_remote_code=True
)
print("Model loaded.")

# Load COCO ground truth
coco_gt = COCO(ANN_FILE)
categories = coco_gt.loadCats(coco_gt.getCatIds())
category_mapping = {c["name"]: c["id"] for c in categories}
print(f"COCO categories loaded: {len(category_mapping)}")

# Get normalization parameters from processor (needed for attack)
IMG_MEAN = torch.tensor(processor.image_processor.image_mean, device=device, dtype=torch_dtype).view(1, 3, 1, 1)
IMG_STD = torch.tensor(processor.image_processor.image_std, device=device, dtype=torch_dtype).view(1, 3, 1, 1)

# Load image file list
files = sorted(os.listdir(IMAGE_DIR))
if NUM_IMAGES is not None:
    files = files[:NUM_IMAGES]
print(f"Will process {len(files)} images.")

## 4. Core Utilities: NMS and Inference

A **single, standard inference function** used by ALL conditions (clean, attacked, defended).  
No score recomputation. No multi-scale. No prompt ensemble. Just the model's raw output.

In [ ]:
# ============================================================
# NMS -- Standard non-maximum suppression
# ============================================================

def box_iou(a, b):
    """Compute IoU between two boxes [x1,y1,x2,y2]."""
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0


def non_max_suppression(boxes, labels, scores, iou_thr=0.5):
    """Standard NMS: suppress overlapping boxes of the same class."""
    if not boxes:
        return [], [], []
    boxes = np.array(boxes)
    idxs = np.argsort(scores)[::-1]
    keep, keep_labels, keep_scores = [], [], []
    for i in idxs:
        suppress = False
        for j in keep:
            if labels[i] == labels[j] and box_iou(boxes[i], boxes[j]) > iou_thr:
                suppress = True
                break
        if not suppress:
            keep.append(i)
            keep_labels.append(labels[i])
            keep_scores.append(scores[i])
    return boxes[keep].tolist(), keep_labels, keep_scores


# ============================================================
# Standard Inference -- Used for ALL conditions
# ============================================================

def run_inference(pil_img):
    """
    Run Florence-2 object detection on a PIL image.
    Single <OD> prompt. Raw model scores. Standard NMS.
    Returns list of COCO-format detections (without image_id).
    """
    with torch.no_grad():
        inputs = processor(text="<OD>", images=pil_img, return_tensors="pt")
        input_ids = inputs.input_ids.to(device)
        pixel_values = inputs.pixel_values.to(device=device, dtype=torch_dtype)

        gen_ids = model.generate(
            input_ids=input_ids,
            pixel_values=pixel_values,
            max_new_tokens=512,
            num_beams=5,
            do_sample=False,
        )
        txt = processor.batch_decode(gen_ids, skip_special_tokens=False)[0]
        parsed = processor.post_process_generation(
            txt, task="<OD>", image_size=(pil_img.width, pil_img.height)
        ) or {}

    od = parsed.get("<OD>", {})
    bboxes = od.get("bboxes", [])
    labels = od.get("labels", [])
    # Florence-2 OD does not output scores; use 1.0 as uniform confidence.
    # This is consistent -- same for clean, attacked, and defended.
    scores = [1.0] * len(bboxes)

    # Apply standard NMS
    kept_boxes, kept_labels, kept_scores = non_max_suppression(
        bboxes, labels, scores, iou_thr=NMS_IOU_THRESHOLD
    )

    # Format for COCO evaluation
    results = []
    for box, label, score in zip(kept_boxes, kept_labels, kept_scores):
        cid = category_mapping.get(label)
        if cid is None:
            continue
        x1, y1, x2, y2 = box
        w, h = x2 - x1, y2 - y1
        if w <= 0 or h <= 0:
            continue
        results.append({
            "bbox": [x1, y1, w, h],
            "category_id": cid,
            "score": score,
        })
    return results


print("Inference pipeline ready.")

## 5. FGSM Attack

Clean implementation: computes gradient of the autoregressive loss w.r.t. input pixels, then applies a single-step perturbation. **No defense logic mixed in.**

Reference: Goodfellow et al., "Explaining and Harnessing Adversarial Examples," ICLR 2015.

In [ ]:
# ============================================================
# FGSM Attack -- Clean implementation
# ============================================================

def fgsm_attack(pil_img, eps=0.01):
    """
    Generate an adversarial image using FGSM.

    1. Process the clean image through Florence-2's processor.
    2. Get target labels via greedy generation (the model's own predictions).
    3. Compute loss gradient w.r.t. pixel_values.
    4. Perturb: x_adv = x + eps * sign(grad).
    5. Convert back to PIL image at original resolution.

    Returns: adversarial PIL image (same size as input).
    """
    orig_size = pil_img.size  # (W, H)

    # Step 1: Process image
    inputs = processor(text="<OD>", images=pil_img, return_tensors="pt")
    input_ids = inputs.input_ids.to(device)
    pixel_values = inputs.pixel_values.to(device=device, dtype=torch_dtype)

    # Step 2: Get target labels (model's own predictions on clean image)
    with torch.no_grad():
        target_ids = model.generate(
            input_ids=input_ids,
            pixel_values=pixel_values,
            max_new_tokens=512,
            num_beams=5,
            do_sample=False,
        )
    # Truncate if too long
    if target_ids.size(1) > 512:
        target_ids = target_ids[:, :512].contiguous()

    # Step 3: Compute gradient
    pixel_values_adv = pixel_values.clone().detach().requires_grad_(True)
    outputs = model(input_ids=input_ids, pixel_values=pixel_values_adv, labels=target_ids)
    loss = outputs.loss
    loss.backward()

    grad_sign = pixel_values_adv.grad.sign()

    # Step 4: Perturb in normalized pixel space
    adv_pixel_values = pixel_values.detach() + eps * grad_sign
    # Clamp to valid normalized range (approximately [-2, 2] for ImageNet normalization)
    adv_pixel_values = torch.clamp(adv_pixel_values, -2.5, 2.5)

    # Step 5: Convert adversarial tensor back to PIL image
    # Denormalize: pixel = tensor * std + mean
    adv_denorm = adv_pixel_values.squeeze(0) * IMG_STD.squeeze(0) + IMG_MEAN.squeeze(0)
    adv_denorm = torch.clamp(adv_denorm, 0.0, 1.0)
    adv_np = (adv_denorm.permute(1, 2, 0).cpu().float().numpy() * 255).astype(np.uint8)
    adv_pil = Image.fromarray(adv_np)

    # Resize back to original dimensions if processor changed size
    if adv_pil.size != orig_size:
        adv_pil = adv_pil.resize(orig_size, Image.BICUBIC)

    return adv_pil


print("FGSM attack function ready.")

## 6. Defense Functions

Each defense takes a PIL image and returns a PIL image. They are applied **after** the attack generates the adversarial image and **before** the image is fed to Florence-2 for inference. This is the correct order.

All defenses are backed by published research:
- **JPEG**: Dziugaite et al., 2016 -- lossy compression removes high-frequency adversarial perturbations.
- **Gaussian Blur**: Xu et al., NDSS 2018 (Feature Squeezing) -- low-pass filter smooths adversarial noise.
- **Median Filter**: Xu et al., NDSS 2018 -- non-linear filter, effective at removing salt-and-pepper-like perturbations while preserving edges.
- **DiffPure**: Nie et al., ICML 2022 -- diffusion forward (add noise) then reverse (denoise) projects image back onto natural manifold.
- **SVD Filtering**: Inspired by Darabi et al., arXiv:2502.14976, 2025 (EigenShield) -- removes singular components that carry adversarial signal.

In [ ]:
# ============================================================
# Defense 1: JPEG Compression
# Reference: Dziugaite et al., "A Study of the Effect of JPG
#            Compression on Adversarial Images," 2016.
# ============================================================

def defend_jpeg(pil_img, quality=JPEG_QUALITY):
    """
    Save image as JPEG at given quality and re-load.
    JPEG's lossy DCT compression removes high-frequency perturbations.
    """
    buffer = BytesIO()
    pil_img.save(buffer, format="JPEG", quality=quality)
    buffer.seek(0)
    return Image.open(buffer).convert("RGB")


# ============================================================
# Defense 2: Gaussian Blur
# Reference: Xu et al., "Feature Squeezing: Detecting
#            Adversarial Examples in DNNs," NDSS 2018.
# ============================================================

def defend_blur(pil_img, sigma=BLUR_SIGMA):
    """
    Apply Gaussian blur. Low-pass filter that smooths
    high-frequency adversarial perturbations.
    """
    return pil_img.filter(ImageFilter.GaussianBlur(radius=sigma))


# ============================================================
# Defense 3: Median Filter
# Reference: Xu et al., "Feature Squeezing," NDSS 2018.
# ============================================================

def defend_median(pil_img, kernel_size=MEDIAN_KERNEL):
    """
    Apply median filter. Non-linear filter effective at
    removing impulse-like adversarial noise while preserving edges.
    """
    return pil_img.filter(ImageFilter.MedianFilter(size=kernel_size))


# ============================================================
# Defense 4: DiffPure (Diffusion-Based Purification)
# Reference: Nie et al., "Diffusion Models for Adversarial
#            Purification," ICML 2022.
# ============================================================

diffpure_model = None  # Lazy-loaded

def _load_diffpure_model():
    """Load a pretrained DDPM for diffusion purification."""
    global diffpure_model
    if diffpure_model is not None:
        return diffpure_model
    
    print("Loading DiffPure DDPM model (google/ddpm-ema-church-256)...")
    # Using a 256x256 DDPM. Other options:
    # - google/ddpm-ema-celebahq-256 (faces)
    # - google/ddpm-cifar10-32 (small, fast, lower quality)
    scheduler = DDPMScheduler.from_pretrained("google/ddpm-ema-church-256")
    unet = UNet2DModel.from_pretrained("google/ddpm-ema-church-256").to(device)
    unet.eval()
    diffpure_model = (unet, scheduler)
    print("DiffPure model loaded.")
    return diffpure_model


def defend_diffpure(pil_img, t_steps=DIFFPURE_TIMESTEPS):
    """
    Diffusion-based purification:
    1. Resize to 256x256 (DDPM input size).
    2. Forward diffusion: add noise for t_steps.
    3. Reverse diffusion: denoise step by step.
    4. Resize back to original size.

    The forward process pushes the image off the natural-image manifold.
    The reverse process projects it back, removing adversarial perturbations.
    """
    if not DIFFPURE_AVAILABLE:
        return pil_img

    unet, scheduler = _load_diffpure_model()
    orig_size = pil_img.size  # (W, H)

    # Convert to tensor, resize to 256x256
    img_tensor = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
    ])(pil_img).unsqueeze(0).to(device)

    # Scale to [-1, 1] as expected by DDPM
    img_tensor = img_tensor * 2.0 - 1.0

    # Forward diffusion: add noise at timestep t
    scheduler.set_timesteps(1000)  # Full schedule
    noise = torch.randn_like(img_tensor)
    timestep = torch.tensor([t_steps], device=device, dtype=torch.long)
    noisy_img = scheduler.add_noise(img_tensor, noise, timestep)

    # Reverse diffusion: denoise from t_steps back to 0
    scheduler.set_timesteps(t_steps)
    sample = noisy_img
    for t in scheduler.timesteps:
        with torch.no_grad():
            noise_pred = unet(sample, t).sample
        sample = scheduler.step(noise_pred, t, sample).prev_sample

    # Convert back to PIL
    sample = (sample.squeeze(0).clamp(-1, 1) + 1.0) / 2.0  # [0, 1]
    sample_np = (sample.permute(1, 2, 0).cpu().float().numpy() * 255).astype(np.uint8)
    purified = Image.fromarray(sample_np).resize(orig_size, Image.BICUBIC)

    return purified


# ============================================================
# Defense 5: SVD Spectral Filtering
# Inspired by: Darabi et al., "EigenShield: Causal Subspace
#   Filtering via Random Matrix Theory for Adversarially
#   Robust VLMs," arXiv:2502.14976, 2025.
#
# Simplified version: truncated SVD on image patches removes
# high-frequency singular components that carry adversarial signal.
# ============================================================

def defend_svd(pil_img, keep_ratio=SVD_KEEP_RATIO):
    """
    SVD-based spectral filtering per channel:
    1. Compute SVD of each color channel.
    2. Keep only top keep_ratio fraction of singular values.
    3. Reconstruct: this removes the smallest singular components
       which tend to capture adversarial high-frequency perturbations.
    """
    img_np = np.array(pil_img).astype(np.float32)
    result = np.zeros_like(img_np)

    for c in range(3):  # Per channel
        channel = img_np[:, :, c]
        U, S, Vt = np.linalg.svd(channel, full_matrices=False)
        # Keep top-k singular values
        k = max(1, int(len(S) * keep_ratio))
        S_filtered = S.copy()
        S_filtered[k:] = 0  # Zero out smallest singular values
        result[:, :, c] = (U * S_filtered) @ Vt

    result = np.clip(result, 0, 255).astype(np.uint8)
    return Image.fromarray(result)


# ============================================================
# Defense registry -- maps name to function
# ============================================================

DEFENSES = {}
if RUN_JPEG_DEFENSE:
    DEFENSES["jpeg"] = defend_jpeg
if RUN_BLUR_DEFENSE:
    DEFENSES["blur"] = defend_blur
if RUN_MEDIAN_DEFENSE:
    DEFENSES["median"] = defend_median
if RUN_DIFFPURE_DEFENSE:
    DEFENSES["diffpure"] = defend_diffpure
if RUN_SVD_DEFENSE:
    DEFENSES["svd"] = defend_svd

print(f"Defenses registered: {list(DEFENSES.keys())}")

## 7. COCO Evaluation Helper

In [ ]:
# ============================================================
# COCO Evaluation -- Save results and compute mAP
# ============================================================

def evaluate_coco(results_list, tag="eval"):
    """
    Save detection results to JSON and run COCO evaluation.
    Returns the 12-element stats array (AP, AP50, AP75, ...).
    Returns all zeros if no detections.
    """
    if not results_list:
        print(f"  [{tag}] No detections -- returning zero mAP.")
        return np.zeros(12)

    out_path = os.path.join(OUTPUT_DIR, f"{tag}.json")
    with open(out_path, "w") as f:
        json.dump(results_list, f)

    coco_dt = coco_gt.loadRes(out_path)
    coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()
    return coco_eval.stats  # [AP, AP50, AP75, AP_S, AP_M, AP_L, AR1, AR10, AR100, AR_S, AR_M, AR_L]


print("COCO evaluation helper ready.")

## 8. Main Evaluation Pipeline

Runs every condition through the **same** `run_inference()` function.  
The only variable is the image fed in:

| Condition | Image fed to `run_inference()` |
|---|---|
| Clean | Original PIL image |
| Clean + Defense X | `defense_x(original_image)` |
| FGSM attacked | `fgsm_attack(original_image)` |
| FGSM + Defense X | `defense_x(fgsm_attack(original_image))` |

In [ ]:
# ============================================================
# Main Evaluation Loop
# ============================================================

def run_full_evaluation():
    """
    Run the complete evaluation:
    1. Clean baseline
    2. Clean + each defense (defense cost)
    3. For each epsilon:
       a. FGSM attacked (no defense)
       b. FGSM + each defense
    """
    start_time = time.time()

    # Result collectors: {condition_tag: [list of COCO detections]}
    all_results = {}
    
    # Conditions to run
    conditions = ["clean"]
    for dname in DEFENSES:
        conditions.append(f"clean+{dname}")
    for eps in EPSILONS:
        eps_tag = f"fgsm_eps{eps}"
        conditions.append(eps_tag)
        for dname in DEFENSES:
            conditions.append(f"{eps_tag}+{dname}")
    
    for cond in conditions:
        all_results[cond] = []
    
    print(f"Conditions to evaluate: {len(conditions)}")
    for c in conditions:
        print(f"  - {c}")
    print()

    # Process each image
    for fname in tqdm(files, desc="Processing images"):
        img_id = int(os.path.splitext(fname)[0])
        img_path = os.path.join(IMAGE_DIR, fname)
        pil_img = Image.open(img_path).convert("RGB")

        # --- Clean baseline ---
        clean_dets = run_inference(pil_img)
        for d in clean_dets:
            d["image_id"] = img_id
        all_results["clean"].extend(clean_dets)

        # --- Clean + each defense (defense cost on clean images) ---
        for dname, dfunc in DEFENSES.items():
            defended_clean = dfunc(pil_img)
            dets = run_inference(defended_clean)
            for d in dets:
                d["image_id"] = img_id
            all_results[f"clean+{dname}"].extend(dets)

        # --- For each epsilon ---
        for eps in EPSILONS:
            eps_tag = f"fgsm_eps{eps}"

            # Generate adversarial image (once per epsilon)
            adv_img = fgsm_attack(pil_img, eps=eps)

            # Attacked, no defense
            adv_dets = run_inference(adv_img)
            for d in adv_dets:
                d["image_id"] = img_id
            all_results[eps_tag].extend(adv_dets)

            # Attacked + each defense
            for dname, dfunc in DEFENSES.items():
                defended_adv = dfunc(adv_img)
                def_dets = run_inference(defended_adv)
                for d in def_dets:
                    d["image_id"] = img_id
                all_results[f"{eps_tag}+{dname}"].extend(def_dets)

        # Memory cleanup
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    elapsed = time.time() - start_time
    print(f"\nAll inference done in {elapsed/60:.1f} minutes.")
    return all_results


# Run it
all_results = run_full_evaluation()

## 9. COCO Evaluation for All Conditions

In [ ]:
# ============================================================
# Run COCO evaluation for every condition
# ============================================================

eval_stats = {}  # {condition_tag: stats_array}

for tag, results_list in all_results.items():
    print(f"\n{'='*60}")
    print(f"Evaluating: {tag} ({len(results_list)} detections)")
    print(f"{'='*60}")
    stats = evaluate_coco(results_list, tag=tag)
    eval_stats[tag] = stats

print("\nAll evaluations complete.")

## 10. Results Summary Table

The key table: for each defense, report **clean cost** (mAP drop on clean images) and **recovery** under each attack strength.

In [ ]:
# ============================================================
# Build results summary
# ============================================================

clean_ap = eval_stats["clean"][0]  # mAP @ IoU 0.50:0.95
clean_ap50 = eval_stats["clean"][1]

print("=" * 90)
print(f"{'FGSM ATTACK & DEFENSE RESULTS':^90}")
print("=" * 90)
print(f"\nClean Baseline:  mAP = {clean_ap:.4f},  AP50 = {clean_ap50:.4f}")
print()

# --- Table 1: Defense cost on clean images ---
print("-" * 60)
print("TABLE 1: Defense Cost (on CLEAN images, no attack)")
print("-" * 60)
print(f"  {'Condition':<25} {'mAP':>8} {'AP50':>8} {'mAP Drop':>10}")
print(f"  {'clean (baseline)':<25} {clean_ap:>8.4f} {clean_ap50:>8.4f} {'---':>10}")
for dname in DEFENSES:
    tag = f"clean+{dname}"
    if tag in eval_stats:
        ap = eval_stats[tag][0]
        ap50 = eval_stats[tag][1]
        drop = clean_ap - ap
        print(f"  {'clean + ' + dname:<25} {ap:>8.4f} {ap50:>8.4f} {drop:>+10.4f}")
print()

# --- Table 2: Attack impact and defense recovery ---
print("-" * 90)
print("TABLE 2: Attack Impact & Defense Recovery")
print("-" * 90)
for eps in EPSILONS:
    eps_tag = f"fgsm_eps{eps}"
    atk_ap = eval_stats[eps_tag][0]
    atk_drop = clean_ap - atk_ap

    print(f"\n  FGSM eps={eps}:")
    print(f"    {'Condition':<30} {'mAP':>8} {'AP50':>8} {'Recovery':>10} {'Recovery%':>12}")
    print(f"    {'attacked (no defense)':<30} {atk_ap:>8.4f} {eval_stats[eps_tag][1]:>8.4f} {'---':>10} {'---':>12}")

    for dname in DEFENSES:
        def_tag = f"{eps_tag}+{dname}"
        if def_tag in eval_stats:
            def_ap = eval_stats[def_tag][0]
            def_ap50 = eval_stats[def_tag][1]
            recovery = def_ap - atk_ap
            recovery_pct = (recovery / atk_drop * 100) if atk_drop > 0 else 0
            print(f"    {'attacked + ' + dname:<30} {def_ap:>8.4f} {def_ap50:>8.4f} {recovery:>+10.4f} {recovery_pct:>11.1f}%")

print()
print("=" * 90)

# --- Save summary to JSON ---
summary = {
    "clean_mAP": float(clean_ap),
    "clean_AP50": float(clean_ap50),
    "defense_cost": {},
    "attack_results": {},
}
for dname in DEFENSES:
    tag = f"clean+{dname}"
    if tag in eval_stats:
        summary["defense_cost"][dname] = {
            "mAP": float(eval_stats[tag][0]),
            "mAP_drop": float(clean_ap - eval_stats[tag][0]),
        }
for eps in EPSILONS:
    eps_tag = f"fgsm_eps{eps}"
    atk_ap = eval_stats[eps_tag][0]
    atk_drop = clean_ap - atk_ap
    entry = {"attacked_mAP": float(atk_ap), "defenses": {}}
    for dname in DEFENSES:
        def_tag = f"{eps_tag}+{dname}"
        if def_tag in eval_stats:
            def_ap = eval_stats[def_tag][0]
            recovery = def_ap - atk_ap
            recovery_pct = (recovery / atk_drop * 100) if atk_drop > 0 else 0
            entry["defenses"][dname] = {
                "mAP": float(def_ap),
                "recovery": float(recovery),
                "recovery_pct": float(recovery_pct),
            }
    summary["attack_results"][str(eps)] = entry

with open(os.path.join(OUTPUT_DIR, "summary.json"), "w") as f:
    json.dump(summary, f, indent=2)
print(f"Summary saved to {OUTPUT_DIR}/summary.json")

## 11. Visualization

In [ ]:
# ============================================================
# Plot 1: mAP across epsilon for each defense
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: mAP vs epsilon
ax = axes[0]
eps_vals = EPSILONS

# Plot attacked (no defense) line
atk_maps = [eval_stats[f"fgsm_eps{e}"][0] for e in eps_vals]
ax.plot(eps_vals, atk_maps, 'r-o', linewidth=2, markersize=8, label="Attacked (no defense)")

# Plot each defense
colors = ['blue', 'green', 'orange', 'purple', 'brown']
for i, dname in enumerate(DEFENSES):
    def_maps = [eval_stats.get(f"fgsm_eps{e}+{dname}", np.zeros(12))[0] for e in eps_vals]
    ax.plot(eps_vals, def_maps, f'-s', color=colors[i % len(colors)],
            linewidth=2, markersize=7, label=f"+ {dname}")

# Clean baseline
ax.axhline(y=clean_ap, color='gray', linestyle='--', linewidth=1.5, label=f"Clean ({clean_ap:.3f})")

ax.set_xlabel("FGSM Epsilon", fontsize=12)
ax.set_ylabel("mAP", fontsize=12)
ax.set_title("FGSM Attack: mAP vs Epsilon", fontsize=14)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Right: Defense cost on clean images (bar chart)
ax = axes[1]
defense_names = list(DEFENSES.keys())
clean_costs = [clean_ap - eval_stats.get(f"clean+{d}", np.zeros(12))[0] for d in defense_names]
clean_maps = [eval_stats.get(f"clean+{d}", np.zeros(12))[0] for d in defense_names]

bars = ax.bar(defense_names, clean_maps, color=colors[:len(defense_names)], alpha=0.8)
ax.axhline(y=clean_ap, color='gray', linestyle='--', linewidth=1.5, label=f"Clean baseline ({clean_ap:.3f})")

for bar, cost in zip(bars, clean_costs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f"-{cost:.3f}", ha='center', va='bottom', fontsize=9)

ax.set_ylabel("mAP", fontsize=12)
ax.set_title("Defense Cost on Clean Images", fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "results_plot.png"), dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to {OUTPUT_DIR}/results_plot.png")

In [ ]:
# ============================================================
# Plot 2: Visual comparison -- sample images
# ============================================================

# Pick a sample image for visualization
sample_fname = files[0]
sample_img = Image.open(os.path.join(IMAGE_DIR, sample_fname)).convert("RGB")
sample_adv = fgsm_attack(sample_img, eps=0.03)

# Build grid: original, adversarial, diff, then each defense
defense_names = list(DEFENSES.keys())
ncols = 3 + len(defense_names)
fig, axes = plt.subplots(1, ncols, figsize=(4 * ncols, 4))

axes[0].imshow(sample_img)
axes[0].set_title("Original")
axes[0].axis("off")

axes[1].imshow(sample_adv)
axes[1].set_title("FGSM (eps=0.03)")
axes[1].axis("off")

# Perturbation difference (magnified)
diff = np.abs(np.array(sample_img).astype(float) - np.array(sample_adv).astype(float))
diff = np.clip(diff * 10, 0, 255).astype(np.uint8)
axes[2].imshow(diff)
axes[2].set_title("Difference (x10)")
axes[2].axis("off")

for i, dname in enumerate(defense_names):
    defended = DEFENSES[dname](sample_adv)
    axes[3 + i].imshow(defended)
    axes[3 + i].set_title(f"+ {dname}")
    axes[3 + i].axis("off")

plt.suptitle("FGSM Attack and Defense Visual Comparison", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "visual_comparison.png"), dpi=150, bbox_inches='tight')
plt.show()

## 12. Net Gain Analysis

A defense is only worth using if: `defended_mAP > max(clean_with_defense_mAP, attacked_mAP)`.

If the defense hurts clean performance more than it helps under attack, the **net gain is negative**.

In [ ]:
# ============================================================
# Net Gain: Is the defense worth it?
# ============================================================

print("=" * 80)
print(f"{'NET GAIN ANALYSIS':^80}")
print("=" * 80)
print()
print("Net Gain = defended_mAP - max(clean+defense_mAP, attacked_mAP)")
print("Positive = defense is helpful. Negative = defense makes things worse.\n")

for eps in EPSILONS:
    eps_tag = f"fgsm_eps{eps}"
    atk_ap = eval_stats[eps_tag][0]

    print(f"  FGSM eps={eps} (attacked mAP = {atk_ap:.4f}):")
    for dname in DEFENSES:
        def_tag = f"{eps_tag}+{dname}"
        clean_def_tag = f"clean+{dname}"
        if def_tag in eval_stats and clean_def_tag in eval_stats:
            def_ap = eval_stats[def_tag][0]
            clean_def_ap = eval_stats[clean_def_tag][0]
            # The "floor" is the better of attacked-no-defense or clean+defense
            floor = max(atk_ap, clean_def_ap)
            net_gain = def_ap - floor
            verdict = "HELPFUL" if net_gain > 0 else "NOT HELPFUL"
            print(f"    {dname:<15} defended_mAP={def_ap:.4f}  "
                  f"floor={floor:.4f}  net_gain={net_gain:+.4f}  [{verdict}]")
    print()

## References

1. Goodfellow, I. et al. "Explaining and Harnessing Adversarial Examples." ICLR, 2015. *(FGSM)*
2. Dziugaite, G. et al. "A Study of the Effect of JPG Compression on Adversarial Images." arXiv:1608.00853, 2016. *(JPEG defense)*
3. Xu, W. et al. "Feature Squeezing: Detecting Adversarial Examples in DNNs." NDSS, 2018. *(Gaussian blur, median filter)*
4. Nie, W. et al. "Diffusion Models for Adversarial Purification." ICML, 2022. *(DiffPure)*
5. Darabi, N. et al. "EigenShield: Causal Subspace Filtering via Random Matrix Theory for Adversarially Robust VLMs." arXiv:2502.14976, 2025. *(SVD filtering)*
6. Xiao, B. et al. "Florence-2: Advancing a Unified Representation for a Variety of Vision Tasks." CVPR, 2024. *(Model)*
7. Fu, X. & Zhang, L. "Adversarial Defense in Vision-Language Models: An Overview." arXiv:2601.12443, 2026. *(Defense taxonomy)*
8. Zhao, J. et al. "One Object, Multiple Lies: Cross-task Adversarial Attack on Unified VLMs." arXiv:2507.07709, 2025. *(Cross-task attacks on Florence-2)*